# RAG Agent Foundation Walkthrough

## Goal

This tutorial verifies the new project contracts without calling external services or requiring credentials. It exercises configuration, Groq model profiles through a fake transport, durable job envelopes, evidence-bearing responses, side-effect approvals, and FastAPI health endpoints.

Run it from the repository's locked `uv` environment. No cell reads `.env`, displays keys, contacts Groq, or connects to Supabase/Redis.

## Setup

From the repository root, install the locked workspace with `uv sync --locked --all-packages --dev`. The notebook expects the editable workspace packages from that environment.

In [1]:
from collections.abc import AsyncIterator
from datetime import UTC, datetime, timedelta
from importlib.metadata import version
from pathlib import Path
from typing import Any
from uuid import UUID

import httpx
from langchain_core.messages import AIMessage, AIMessageChunk
from pydantic import BaseModel, ConfigDict, SecretStr

from rag_api.main import create_app
from rag_core.config import AppSettings
from rag_core.jobs.models import JobEnvelope, JobOperation, JobStatus, ensure_status_transition
from rag_core.models.contracts import ChatMessage, ModelRequest
from rag_core.models.groq import GroqChatModelProvider
from rag_core.retrieval.models import AnswerStatus, Citation, GroundedAnswer
from rag_core.tools.models import ApprovalGrant, ToolCallIntent, ToolEffect

repo_root = next(
    (candidate for candidate in (Path.cwd(), *Path.cwd().parents) if (candidate / 'uv.lock').exists()),
    None,
)
if repo_root is None:
    raise RuntimeError('Run this notebook from inside the repository after `uv sync`.')

{
    'python_workspace': repo_root.name,
    'rag-core': version('rag-core'),
    'rag-api': version('rag-api'),
    'langchain-groq': version('langchain-groq'),
}

{'python_workspace': 'LangChain-RAG-assistant',
 'rag-core': '0.1.0',
 'rag-api': '0.1.0',
 'langchain-groq': '1.1.3'}

## Steps

### 1. Inspect capability-based Groq profiles

Application code selects a capability alias rather than scattering provider model IDs across graphs. The dummy key below is never sent anywhere and remains redacted by Pydantic.

In [2]:
settings = AppSettings(_env_file=None, groq_api_key=SecretStr('demo-key-not-valid'))
profiles = settings.model_profiles()

{
    alias: {
        'provider': profile.provider,
        'model': profile.model,
        'strict_json_schema': profile.supports_strict_structured_output,
        'tool_calling': profile.supports_tool_calling,
    }
    for alias, profile in profiles.items()
}

{'fast_structured': {'provider': 'groq',
  'model': 'openai/gpt-oss-20b',
  'strict_json_schema': True,
  'tool_calling': False},
 'quality': {'provider': 'groq',
  'model': 'openai/gpt-oss-120b',
  'strict_json_schema': False,
  'tool_calling': False},
 'agent': {'provider': 'groq',
  'model': 'openai/gpt-oss-120b',
  'strict_json_schema': False,
  'tool_calling': True}}

### 2. Build a versioned, identifier-only job envelope

Redis transports this small envelope. PostgreSQL owns job status, attempts, leases, cancellation, and dispatch intent.

In [3]:
job = JobEnvelope(
    job_id=UUID('11111111-1111-4111-8111-111111111111'),
    operation=JobOperation.DOCUMENT_INGESTION,
    idempotency_key='document-version:pipeline-v1',
    traceparent='00-0123456789abcdef0123456789abcdef-0123456789abcdef-01',
    enqueued_at=datetime(2026, 8, 29, 12, 0, tzinfo=UTC),
)
ensure_status_transition(JobStatus.QUEUED, JobStatus.RUNNING)
job.model_dump(mode='json')

{'schema_version': 1,
 'job_id': '11111111-1111-4111-8111-111111111111',
 'operation': 'document_ingestion',
 'idempotency_key': 'document-version:pipeline-v1',
 'traceparent': '00-0123456789abcdef0123456789abcdef-0123456789abcdef-01',
 'enqueued_at': '2026-08-29T12:00:00Z'}

### 3. Construct a grounded response

A fully answered RAG response must carry at least one evidence citation. Unsupported answers fail validation rather than looking successful.

In [4]:
grounded_answer = GroundedAnswer(
    status=AnswerStatus.ANSWERED,
    text='PostgreSQL is the authoritative source for durable job state.',
    confidence=0.96,
    citations=(
        Citation(
            ordinal=1,
            chunk_id=UUID('22222222-2222-4222-8222-222222222222'),
            document_id=UUID('33333333-3333-4333-8333-333333333333'),
            source_label='ADR-005: PostgreSQL authoritative job state',
        ),
    ),
)
grounded_answer.model_dump(mode='json')

{'status': 'answered',
 'text': 'PostgreSQL is the authoritative source for durable job state.',
 'citations': [{'ordinal': 1,
   'chunk_id': '22222222-2222-4222-8222-222222222222',
   'document_id': '33333333-3333-4333-8333-333333333333',
   'source_label': 'ADR-005: PostgreSQL authoritative job state'}],
 'confidence': 0.96,
 'diagnostic_codes': []}

### 4. Bind human approval to one exact side effect

A write approval is tied to the user, workspace, invocation, exact argument digest, and expiry. Changing an argument invalidates it.

In [5]:
approval_time = datetime(2026, 8, 29, 12, 0, tzinfo=UTC)
tool_intent = ToolCallIntent(
    invocation_id=UUID('44444444-4444-4444-8444-444444444444'),
    workspace_id=UUID('55555555-5555-4555-8555-555555555555'),
    user_id=UUID('66666666-6666-4666-8666-666666666666'),
    connector='google_calendar',
    action='create_event',
    effect=ToolEffect.SIDE_EFFECT,
    arguments={'title': 'Architecture review', 'hour': 14},
    idempotency_key='calendar:create:architecture-review',
)
approval = ApprovalGrant(
    approval_id=UUID('77777777-7777-4777-8777-777777777777'),
    invocation_id=tool_intent.invocation_id,
    workspace_id=tool_intent.workspace_id,
    user_id=tool_intent.user_id,
    arguments_sha256=tool_intent.arguments_sha256,
    expires_at=approval_time + timedelta(minutes=5),
)
changed_intent = tool_intent.model_copy(update={'arguments': {'title': 'Changed event'}})
{
    'exact_intent_authorized': approval.authorizes(tool_intent, at=approval_time),
    'changed_intent_authorized': approval.authorizes(changed_intent, at=approval_time),
}

{'exact_intent_authorized': True, 'changed_intent_authorized': False}

### 5. Exercise the Groq boundary without a live API call

The fake below behaves like the LangChain chat runtime. The production adapter still enforces strict JSON Schema for structured decisions and normalizes provider metadata and streaming chunks.

In [6]:
class RouteDecision(BaseModel):
    model_config = ConfigDict(extra='forbid')

    route: str
    reason: str


class DemoStructuredModel:
    async def ainvoke(self, _: Any) -> dict[str, str]:
        return {'route': 'rag', 'reason': 'The request asks about indexed project evidence.'}


class DemoGroqRuntime:
    async def ainvoke(self, _: Any) -> AIMessage:
        return AIMessage(
            content='Use the deterministic RAG path.',
            usage_metadata={'input_tokens': 11, 'output_tokens': 6, 'total_tokens': 17},
            response_metadata={
                'model_name': 'openai/gpt-oss-120b',
                'finish_reason': 'stop',
                'x_groq': {'id': 'demo-request-id'},
            },
        )

    async def astream(self, _: Any) -> AsyncIterator[AIMessageChunk]:
        yield AIMessageChunk(content='Use deterministic ')
        yield AIMessageChunk(content='RAG.', response_metadata={'finish_reason': 'stop'})

    def with_structured_output(self, _: Any, /, **__: Any) -> DemoStructuredModel:
        return DemoStructuredModel()


demo_runtime = DemoGroqRuntime()
provider = GroqChatModelProvider(
    api_key=SecretStr('demo-key-not-valid'),
    profiles=profiles,
    model_factory=lambda _profile, _api_key: demo_runtime,
)
model_request = ModelRequest(
    profile_alias='fast_structured',
    messages=(ChatMessage(role='user', content='Which route should answer this?'),),
)

route = await provider.complete_structured(model_request, RouteDecision)
completion = await provider.complete(model_request.model_copy(update={'profile_alias': 'quality'}))
streamed = [
    chunk
    async for chunk in provider.stream(
        model_request.model_copy(update={'profile_alias': 'quality'})
    )
]
{
    'route': route.model_dump(),
    'completion': completion.model_dump(),
    'stream_text': ''.join(chunk.text for chunk in streamed),
}

{'route': {'route': 'rag',
  'reason': 'The request asks about indexed project evidence.'},
 'completion': {'text': 'Use the deterministic RAG path.',
  'model': 'openai/gpt-oss-120b',
  'finish_reason': 'stop',
  'provider_request_id': 'demo-request-id',
  'usage': {'input_tokens': 11, 'output_tokens': 6, 'total_tokens': 17}},
 'stream_text': 'Use deterministic RAG.'}

### 6. Call FastAPI in-process

The ASGI transport checks the real routes without starting a server. Readiness currently verifies configuration presence; live dependency probes are a later foundation slice.

In [7]:
application = create_app(settings)
transport = httpx.ASGITransport(app=application)
async with httpx.AsyncClient(transport=transport, base_url='http://notebook') as client:
    live_response = await client.get('/health/live')
    ready_response = await client.get('/health/ready')

{
    'liveness': {'status_code': live_response.status_code, **live_response.json()},
    'readiness': {'status_code': ready_response.status_code, **ready_response.json()},
}

HTTP Request: GET http://notebook/health/live "HTTP/1.1 200 OK"


HTTP Request: GET http://notebook/health/ready "HTTP/1.1 200 OK"


{'liveness': {'status_code': 200,
  'status': 'ok',
  'service': 'rag-api',
  'version': '0.1.0'},
 'readiness': {'status_code': 200,
  'status': 'ok',
  'checks': {'database_configured': True,
   'groq_configured': True,
   'redis_configured': True}}}

## Checks

These assertions make the notebook fail loudly if a foundation invariant regresses.

In [8]:
assert route.route == 'rag'
assert completion.provider_request_id == 'demo-request-id'
assert completion.usage.total_tokens == 17
assert ''.join(chunk.text for chunk in streamed) == 'Use deterministic RAG.'
assert approval.authorizes(tool_intent, at=approval_time)
assert not approval.authorizes(changed_intent, at=approval_time)
assert live_response.status_code == 200
assert ready_response.status_code == 200

'All credential-free foundation checks passed.'

'All credential-free foundation checks passed.'

## Next Steps

1. Implement the transactional job/outbox dispatcher and recovery drill.
2. Add Supabase OTP authentication, workspaces, memberships, and tenant RLS.
3. Build the durable ingestion pipeline and first hybrid retrieval baseline.
4. Add opt-in live Groq and container integration notebooks only after development credentials and Docker are available.

Credential inventory and setup links live in `docs/configuration/credentials-and-services.md`; secrets belong in an untracked `.env` or deployment secret manager, never in this notebook.